In [4]:
import pyspark
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [6]:
!wget -O fhvhv_tripdata_2021-01.csv.gz https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-03-07 13:59:43--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 20.26.156.215
Connecting to github.com (github.com)|20.26.156.215|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-07T14%3A41%3A19Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-07T13%3A40%3A47Z&ske=2026-03-07T14%3A41%3A19Z&sks=b&skv=2018-11-09&sig=nRGx8T5OlzSwnc0gtQiTghH6qHBoFYBVqvNLZfsWefA%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3Mjg5NTU4MywibmJmIjoxNzcyODkxOTgzLCJwYXRoIj

In [7]:
!gzip -dk fhvhv_tripdata_2021-01.csv.gz

In [8]:
!wc -l fhvhv_tripdata_2021-01.csv

 11908469 fhvhv_tripdata_2021-01.csv


In [9]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [10]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [11]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [12]:
import pandas as pd

In [13]:
df_pandas = pd.read_csv('head.csv')

In [14]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [18]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

Integer - 4 bytes
Long - 8 bytes

In [19]:
from pyspark.sql import types

In [20]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [32]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [26]:
df = df.repartition(24)

In [27]:
df.write.parquet('fhvhv/2021/01/', mode='overwrite')

26/03/07 14:07:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/07 14:07:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/07 14:07:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/07 14:07:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/07 14:07:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/07 14:07:44 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/07 14:07:44 WARN MemoryManager: Total allocation exceeds 95.

In [28]:
df = spark.read.parquet('fhvhv/2021/01/')

In [29]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- dropoff_datetime: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- SR_Flag: string (nullable = true)



SELECT * FROM df WHERE hvfhs_license_num =  HV0003

In [30]:
from pyspark.sql import functions as F

In [31]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02880|2021-01-02 10:59:19|2021-01-02 11:09:35|         164|          90|   NULL|
|           HV0003|              B02883|2021-01-01 16:58:08|2021-01-01 17:05:04|         189|         106|   NULL|
|           HV0003|              B02872|2021-01-01 19:20:38|2021-01-01 19:25:25|          32|         185|   NULL|
|           HV0005|              B02510|2021-01-02 21:17:59|2021-01-02 21:41:44|          76|         210|   NULL|
|           HV0003|              B02869|2021-01-03 14:41:30|2021-01-03 14:49:41|         233|         170|   NULL|
|           HV0003|              B02875|2021-01-01 09:29:39|2021-01-01 09:51:18|

In [32]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [33]:
crazy_stuff('B02884')

's/b44'

In [34]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [35]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  a/b40| 2021-01-02|  2021-01-02|         164|          90|
|  a/b43| 2021-01-01|  2021-01-01|         189|         106|
|  e/b38| 2021-01-01|  2021-01-01|          32|         185|
|  e/9ce| 2021-01-02|  2021-01-02|          76|         210|
|  e/b35| 2021-01-03|  2021-01-03|         233|         170|
|  e/b3b| 2021-01-01|  2021-01-01|         256|          85|
|  e/9ce| 2021-01-03|  2021-01-03|         127|         236|
|  a/a7a| 2021-01-03|  2021-01-03|         121|          98|
|  e/b3e| 2021-01-01|  2021-01-01|         254|         254|
|  e/9ce| 2021-01-02|  2021-01-02|          60|         168|
|  s/b44| 2021-01-03|  2021-01-03|         226|          82|
|  s/b13| 2021-01-03|  2021-01-03|         215|         130|
|  e/b35| 2021-01-01|  2021-01-01|         114|         140|
|  e/9ce| 2021-01-02|  2

In [36]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003')


DataFrame[pickup_datetime: string, dropoff_datetime: string, PULocationID: string, DOLocationID: string]

In [37]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,


In [38]:
spark.stop()